# PhishARG — Entrenamiento Mejorado del Modelo Hibrido NLP (A100 GPU)

**Proyecto de Tesis:** PhishARG — Deteccion de Phishing con IA Explicable

**Autores:** Gabriel Adia & Tomas Basualdo | **Universidad:** UADE

Este notebook entrena el clasificador hibrido **Sentence Transformers (MiniLM) + XGBoost** con la maxima potencia posible usando la GPU A100 de Google Colab.

### Configuracion mejorada vs entrenamiento local:
| Parametro | Local (CPU) | Colab A100 |
|-----------|-------------|------------|
| Epocas | 1 | **5** |
| Tokens por correo | 128 | **384** |
| Capas entrenables | 2 | **4** |
| Batch size | 16 | **64** |
| Device | CPU | **CUDA (A100)** |
| Tiempo estimado | ~40 min | **~1-2 min** |

> **IMPORTANTE:** Antes de ejecutar, anda a `Entorno de ejecucion` > `Cambiar tipo de entorno de ejecucion` > selecciona **A100 GPU** > **Guardar**.


## Paso 1: Clonar el repositorio

In [ ]:
# Clonar el repositorio de PhishARG
!git clone https://github.com/gabrieladia1979/flujo_v2.git
%cd flujo_v2
!git checkout codex/hybrid-nlp
print("\nRepositorio clonado y rama codex/hybrid-nlp activa.")

## Paso 2: Instalar dependencias y verificar GPU

In [ ]:
!pip install -q sentence-transformers==5.7.0 xgboost scikit-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"Memoria GPU: {mem:.0f} GB")
else:
    print("ERROR: No hay GPU. Activa la GPU en Entorno de ejecucion > Cambiar tipo.")

## Paso 3: Subir el dataset

El archivo `multilingual-v1.csv` (19 MB) esta en tu PC en:

`flujo_v2\artifacts\hybrid\corpus\multilingual-v1.csv`

Ejecuta esta celda y selecciona ese archivo.

In [ ]:
import shutil
from google.colab import files

!mkdir -p artifacts/hybrid/corpus
!mkdir -p reports

print("Selecciona el archivo multilingual-v1.csv desde tu PC:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, "artifacts/hybrid/corpus/" + filename)
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f"Archivo {filename} subido ({size_mb:.1f} MB)")

## Paso 4: Entrenar el modelo con A100 (MAXIMA POTENCIA)

Esta celda entrena con los parametros mas agresivos posibles:
- **5 epocas** — el modelo ve los datos 5 veces completas
- **384 tokens** — lee casi el triple de contexto por correo
- **4 capas entrenables** — ajusta mas profundamente la red neuronal
- **batch-size 64** — aprovecha los 40 GB de VRAM de la A100

> En la A100 esto tarda **~1-2 minutos**.

In [ ]:
# ENTRENAMIENTO MAXIMO - 5 epocas, 384 tokens, 4 capas, batch 64
!python scripts/train_hybrid.py \
  --dataset artifacts/hybrid/corpus/multilingual-v1.csv \
  --phishing-label 1 \
  --output artifacts/hybrid/multilingual-candidate-a100 \
  --report reports/hybrid_multilingual_a100 \
  --epochs 5 \
  --batch-size 64 \
  --max-tokens 384 \
  --trainable-layers 4 \
  --device cuda

## Paso 5: Evaluar y comparar contra el modelo original

Probamos contra los 14 casos dificiles y comparamos con el legacy.

In [ ]:
!python scripts/evaluate_hybrid.py \
  --model artifacts/hybrid/multilingual-candidate-a100 \
  --dataset data/classifier_eval_v1.jsonl \
  --output reports/hybrid_eval_a100 \
  --compare-legacy

## Paso 6: Ver resultados del entrenamiento

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("reports/hybrid_multilingual_a100.json").read_text())

print("=" * 70)
print("RESULTADOS - ENTRENAMIENTO HIBRIDO A100 (MAXIMO)")
print("=" * 70)
print(f"Filas: {report['dataset_audit']['usable_rows']} | Grupos: {report['dataset_audit']['groups']}")
print()

for name, r in report["results"].items():
    m = r["test_metrics"]
    cm = m["confusion_matrix"]
    print(f"  {name}")
    print(f"    F1-Score:  {m['f1']:.4f}")
    print(f"    Precision: {m['precision']:.4f}")
    print(f"    Recall:    {m['recall']:.4f}")
    print(f"    TP: {cm['true_positive']} | TN: {cm['true_negative']} | FP: {cm['false_positive']} | FN: {cm['false_negative']}")
    print()

ft = report.get("fine_tuning", {})
if ft:
    print(f"Ajuste neuronal:")
    print(f"  Pasos totales: {ft.get('total_steps', '?')}")
    params = ft.get("trainable_parameters", 0)
    print(f"  Parametros entrenables: {params:,}")
    print(f"  Pesos modificados: {ft.get('weights_changed', '?')}")

# Mostrar comparativa local vs Colab
print()
print("=" * 70)
print("COMPARATIVA: Local (1 epoca, 128 tokens) vs Colab A100 (5 epocas, 384 tokens)")
print("=" * 70)
print("El modelo local (candidate-v1) obtuvo:")
print("  tfidf_xgboost:                F1 = 0.9614")
print("  frozen_embeddings_xgboost:    F1 = 0.8812")
print("  finetuned_embeddings_xgboost: F1 = 0.9041")
print()
print("El modelo Colab A100 (arriba) deberia superar esos valores.")
print("=" * 70)

## Paso 7: Descargar el modelo entrenado

Descarga el modelo y los reportes como ZIP.

Despues descomprimilo en tu PC en:
`flujo_v2\artifacts\hybrid\multilingual-candidate-a100\`

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("multilingual-candidate-a100", "zip",
                    "artifacts/hybrid/multilingual-candidate-a100")
shutil.make_archive("reports_a100", "zip", "reports")

print("Descargando modelo y reportes...")
files.download("multilingual-candidate-a100.zip")
files.download("reports_a100.zip")

print("\nDescarga completa!")
print("\nPara usarlo en tu PC:")
print("  $env:PHISHARG_HYBRID_MODEL_DIR = 'artifacts/hybrid/multilingual-candidate-a100'")
print("  .venv-hybrid/Scripts/python.exe -m uvicorn main:app --port 8001")
print("\nEndpoint experimental: POST http://127.0.0.1:8001/api/v1/analyze/hybrid")